# Lab Instructions

Choose your own adventure! In this lab, you will select a dataset, identify the target feature, and determine what relationships are present between the target and the other features in the data.

The dataset should have at least 5 features plus the target and at least a few hundred rows.  If the original dataset has more than 5 features, you may select the 5 that seem most interesting for this project. The subject can be anything you choose.  

For your lab submission, describe the dataset and the features - including all of the values of the features - and identify the target feature.  Then make visualizations to show the relationship of each feature to the target.  Which feature(s) seem most related?  Which features don't seem to influence the value of the target?  Draw at least one big picture conclusion about your data from the visualizations you've created.


# Dataset overview

I am using a California Housing Dataset which derives from the 1990 census. Each row represents a census block group, which is the smallest geographic unit that the Census beureau publishes sample data.

### Target feature

My target feature is the median house value in the census block groups. The median house value in the data is capped at $500,000, which will be noted later.

### Input features

* Median income - The median income for the houses in the block.
* Average rooms - The median age of the houses in the block.
* House age - The average number of rooms per household.
* Latitude - The latitude of the block group.
* Average occupancy - How many occupants who live in the household.

### Overall features



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.datasets import fetch_california_housing

raw = fetch_california_housing(as_frame=True)
df_full = raw.frame  # includes target

# Select 5 features + target
df = df_full[['MedInc', 'HouseAge', 'AveRooms', 'Latitude', 'AveOccup', 'MedHouseVal']].copy()
df.columns = ['median_income', 'house_age', 'avg_rooms', 'latitude', 'avg_occupancy', 'median_house_value']

# Convert target from $100k units to full dollars
df['median_house_value'] = df['median_house_value'] * 100_000

print(f'Shape: {df.shape}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['median_house_value'], bins=50, edgecolor='white', alpha=0.7)
axes[0].set_title('Distribution of Median House Value')
axes[0].set_xlabel('Median House Value')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['median_house_value'], vert=True, patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot of Median House Value')
axes[1].set_ylabel('Median House Value')

print(f"Mean:   ${df['median_house_value'].mean():,.0f}")
print(f"Median: ${df['median_house_value'].median():,.0f}")
print(f"Std:    ${df['median_house_value'].std():,.0f}")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(df['median_income'], df['median_house_value'], alpha=0.15, s=8)

ax.set_title('Income vs. House Value')
ax.set_xlabel('Median Income (tens of thousands)')
ax.set_ylabel('Median House Value')

corr = df['median_income'].corr(df['median_house_value'])
print(f'Pearson correlation: {corr:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

df['age_bin'] = pd.cut(df['house_age'], bins=range(0, 55, 5))
age_group = df.groupby('age_bin', observed=True)['median_house_value'].median().reset_index()

ax.bar([str(b) for b in age_group['age_bin']], age_group['median_house_value'])
ax.set_title('Median House Value by House Age Group')
ax.set_xlabel('House Age')
ax.set_ylabel('Median House Value')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(plot_df['avg_rooms'], plot_df['median_house_value'], alpha=0.15, s=8)

ax.set_title('Average Rooms per Household vs. Median House Value')
ax.set_xlabel('Average Rooms per Household')
ax.set_ylabel('Median House Value')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(df['latitude'], df['median_house_value'], alpha=0.12, s=8)

# Highlight LA (34°) and SF (~37.8°) latitude bands
ax.axvline(x=34.05, color='red', linestyle='--', linewidth=1.2, label='Los Angeles (~34°N)')
ax.axvline(x=37.77, color='blue', linestyle='--', linewidth=1.2, label='San Francisco (~37.8°N)')

ax.set_title('Latitude vs. Median House Value')
ax.set_xlabel('Latitude')
ax.set_ylabel('Median House Value')
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Cap extreme outliers for a readable plot (occupancy >10 is likely data noise)
plot_df = df[df['avg_occupancy'] < 10]

# Scatter
axes[0].scatter(plot_df['avg_occupancy'], plot_df['median_house_value'], alpha=0.12, s=8,)
m, b = np.polyfit(plot_df['avg_occupancy'], plot_df['median_house_value'], 1)
x_line = np.linspace(plot_df['avg_occupancy'].min(), plot_df['avg_occupancy'].max(), 100)
axes[0].set_title('Avg Occupancy vs. House Value')
axes[0].set_xlabel('Average Occupancy')
axes[0].set_ylabel('Median House Value')
axes[0].legend()

plot_df = plot_df.copy()
plot_df['occ_bin'] = pd.cut(plot_df['avg_occupancy'], bins=[1, 2, 3, 4, 5, 6, 10], labels=['1–2', '2–3', '3–4', '4–5', '5–6', '6–10'])
occ_group = plot_df.groupby('occ_bin', observed=True)['median_house_value'].median()
axes[1].bar(occ_group.index.astype(str), occ_group.values, edgecolor='white', alpha=0.85)
axes[1].set_title('Median House Value by Occupancy Band')
axes[1].set_xlabel('Average Occupancy')
axes[1].set_ylabel('Median House Value')
plt.show()